# SO101 ACT Training (Kaggle GPU)
Train ACT policy on SO101 dataset using LeRobot framework

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['MUJOCO_GL'] = 'egl'

In [ ]:
!pip install -q lerobot 2>/dev/null || pip install -q 'lerobot[all]' -i https://pypi.tuna.tsinghua.edu.cn/simple

In [ ]:
from huggingface_hub import snapshot_download, HfApi
import os

api = HfApi(token=HF_TOKEN, endpoint='https://hf-mirror.com')

# Download dataset
dataset_path = snapshot_download(
    repo_id=DATASET_REPO,
    repo_type='dataset',
    token=HF_TOKEN,
    local_dir='/kaggle/working/dataset'
)
print(f'Dataset at: {dataset_path}')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(
    repo_id=DATASET_REPO,
    root='/kaggle/working/dataset'
)
print(f'Episodes: {dataset.num_episodes}')
print(f'Frames: {dataset.num_frames}')
print(f'Features: {dataset.features}')

In [ ]:
from lerobot.common.policies.act.configuration_act import ACTConfig
from lerobot.common.policies.act.modeling_act import ACTPolicy

policy_cfg = ACTConfig()
policy_cfg.chunk_size = 10
policy_cfg.temporal_ensemble_coeff = 0.01
policy = ACTPolicy(policy_cfg)
policy.to('cuda')
print('ACT policy created')

In [ ]:
from lerobot.common.optimizers.optimizers import AdamOptimizerConfig
from lerobot.configs.train import TrainPipelineConfig
from lerobot.common.datasets.lerobot_dataset import LeRobotDatasetConfig
from lerobot.common.policies.act.configuration_act import ACTConfig

train_cfg = TrainPipelineConfig(
    dataset=LeRobotDatasetConfig(repo_id=DATASET_REPO, root='/kaggle/working/dataset'),
    policy=ACTConfig(chunk_size=10, temporal_ensemble_coeff=0.01),
    optimizer=AdamOptimizerConfig(lr=1e-4),
    batch_size=16,
    num_workers=4,
    steps=TRAINING_STEPS,
    save_freq=5000,
    log_freq=100,
    eval_freq=5000,
)
print(f'Training config: {TRAINING_STEPS} steps, batch_size=16')

In [ ]:
from lerobot.scripts.train import train

output_dir = '/kaggle/working/outputs/train/so101_act'
os.makedirs(output_dir, exist_ok=True)

train(train_cfg, output_dir=output_dir)
print('Training complete!')

In [ ]:
# Upload model to HuggingFace
from huggingface_hub import HfApi
import glob

api = HfApi(token=HF_TOKEN, endpoint='https://hf-mirror.com')

# Create model repo if not exists
try:
    api.create_repo(repo_id=MODEL_REPO, repo_type='model', exist_ok=True)
except Exception as e:
    print(f'Repo creation: {e}')

# Upload checkpoint
checkpoint_files = glob.glob(f'{output_dir}/**/*', recursive=True)
print(f'Files to upload: {len([f for f in checkpoint_files if os.path.isfile(f)])}')

api.upload_folder(
    folder_path=output_dir,
    repo_id=MODEL_REPO,
    repo_type='model',
    token=HF_TOKEN,
)
print(f'Model uploaded to {MODEL_REPO}')

In [ ]:
print(f'=== Training Complete ===')
print(f'Model: {MODEL_REPO}')
print(f'Steps: {TRAINING_STEPS}')
print(f'Policy: {POLICY_TYPE}')